# Fabric Workspace GUID Extractor
Extracts all artifact GUIDs from one or more Microsoft Fabric workspaces
and optionally updates the Variable Library definition via the Fabric REST API.
**Usage:**
1. Set `WORKSPACE_NAMES` below to target specific workspaces by name, or leave empty to auto-detect the current workspace.
2. Set `UPDATE_VALUE_SETS = True` and configure `VALUE_SETS_TO_UPDATE` to control which value sets get updated.
   Use `"default"` to update the base variables, or specific value set names (e.g. `"Env-1D"`) for environment overrides.
3. Run all cells.


## Configuration

Pick the workspaces to scan, name the target Variable Library, and decide which value sets get written. `"__current__"` is a convenience token that resolves to the workspace this notebook is running in - useful when the same notebook is deployed across dev/test/prod.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────
# Add workspace names here to extract GUIDs from multiple workspaces.
# Leave empty to auto-detect the current workspace via notebookutils.
WORKSPACE_NAMES = ["<WorkspaceName>"]

# Set to True to update the Variable Library via the Fabric REST API
UPDATE_VALUE_SETS = True

# Display name of the Variable Library item in Fabric
VARIABLE_LIBRARY_NAME = "<VariableLibraryName>"

# Which workspace contains the Variable Library to update.
# Must be one of the names in WORKSPACE_NAMES above.
VARIABLE_LIBRARY_WORKSPACE = "__current__"

# Map value set names to workspace names. Use "default" to update the base variables.
# Use "__current__" as the workspace name to auto-resolve to the workspace this notebook runs in.
# Only listed targets are touched; unlisted value sets are passed through unchanged.
VALUE_SETS_TO_UPDATE = {
    "default": "__current__",
    "<SetName1>": "<WorkspaceName>-<EnvironmentSuffix>",
    # "<SetName2>": "<WorkspaceName>-<EnvironmentSuffix>",
    # "<SetName3>": "<WorkspaceName>-<EnvironmentSuffix>",
}

# --- Strict Mode ---
# False (default) - print the summary and continue even if some value sets
#                   failed or were skipped. Right for scheduled runs where the
#                   per-set report is enough signal.
# True  - raise RuntimeError after the summary when any value set failed. Use
#         when running from CI / orchestration and you want the notebook exit
#         code to reflect state.
STRICT = False

# Canonical Tier-2 result buckets - populated by the extract and update loops
# below. Each "name" is a value-set identifier (e.g. "default", "Env-1D") or
# a workspace scan label.
results = {
    "succeeded": [],  # list[str]
    "skipped":   [],  # list[dict]: {"name": str, "reason": str}
    "failed":    [],  # list[dict]: {"name": str, "error": str}
}

## Authentication & Workspace Resolution

Acquires a Fabric token from the notebook runtime, paginates the workspaces API to resolve display names to GUIDs, and defines an LRO poller for later `getDefinition` / `updateDefinition` calls.

In [ ]:
import json
import base64
import time
import requests
from typing import Dict, List, Optional, Tuple

# ── Authenticate via Fabric runtime ──────────────────────────────────────────────
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
TOKEN = notebookutils.credentials.getToken("https://api.fabric.microsoft.com/.default")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


def resolve_workspace_names(names: List[str]) -> List[Tuple[str, str]]:
    """Resolve workspace display names to (workspace_id, workspace_name) pairs via the Fabric API."""
    url = f"{FABRIC_API_BASE}/workspaces"
    all_workspaces = []

    # Paginate through all accessible workspaces
    while url:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            raise RuntimeError(
                f"Failed to list workspaces: {response.status_code} {response.text}"
            )
        data = response.json()
        all_workspaces.extend(data.get("value", []))
        url = data.get("continuationUri")

    # Build a lookup by displayName
    name_lookup = {ws["displayName"]: ws["id"] for ws in all_workspaces}

    resolved = []
    for name in names:
        ws_id = name_lookup.get(name)
        if ws_id:
            resolved.append((ws_id, name))
        else:
            print(f"WARNING: Workspace '{name}' not found. Available workspaces:")
            for available in sorted(name_lookup.keys()):
                print(f"  - {available}")
            raise RuntimeError(f"Workspace '{name}' not found.")

    return resolved


def poll_lro(response, headers: Dict) -> requests.Response:
    """Poll a Fabric long-running operation until completion, then fetch the result."""
    if response.status_code != 202:
        return response

    location = response.headers.get("Location")

    # Poll operation status until no longer Running
    while True:
        retry_after = int(response.headers.get("Retry-After", 5))
        print(f"  Operation in progress, retrying in {retry_after}s...")
        time.sleep(retry_after)
        response = requests.get(location, headers=headers)

        status = response.json().get("status", "")
        if status not in ("NotStarted", "Running"):
            break

    if response.json().get("status") != "Succeeded":
        return response

    # Fetch the actual result from the /result endpoint
    result_url = f"{location}/result"
    return requests.get(result_url, headers=headers)


# ── Resolve current workspace ────────────────────────────────────────────────────
ctx = notebookutils.runtime.context
CURRENT_WORKSPACE_ID = ctx.get("workspaceId") or ctx.get("currentWorkspaceId")
CURRENT_WORKSPACE_NAME = ctx.get("workspaceName") or ctx.get("currentWorkspaceName") or "Unknown"
if not CURRENT_WORKSPACE_ID:
    raise RuntimeError("Could not detect workspace ID from notebookutils.runtime.context")
print(f"Current workspace: {CURRENT_WORKSPACE_NAME} ({CURRENT_WORKSPACE_ID})")

# ── Resolve workspaces ───────────────────────────────────────────────────────────
if WORKSPACE_NAMES:
    WORKSPACES = resolve_workspace_names(WORKSPACE_NAMES)
else:
    WORKSPACES = [(CURRENT_WORKSPACE_ID, CURRENT_WORKSPACE_NAME)]

# Ensure current workspace is in the list (needed for "__current__" resolution)
current_in_list = any(ws_id == CURRENT_WORKSPACE_ID for ws_id, _ in WORKSPACES)
if not current_in_list:
    WORKSPACES.append((CURRENT_WORKSPACE_ID, CURRENT_WORKSPACE_NAME))

# Resolve "__current__" in VALUE_SETS_TO_UPDATE and VARIABLE_LIBRARY_WORKSPACE
VALUE_SETS_TO_UPDATE = {
    k: (CURRENT_WORKSPACE_NAME if v == "__current__" else v)
    for k, v in VALUE_SETS_TO_UPDATE.items()
}
if VARIABLE_LIBRARY_WORKSPACE == "__current__":
    VARIABLE_LIBRARY_WORKSPACE = CURRENT_WORKSPACE_NAME

print(f"Targeting {len(WORKSPACES)} workspace(s):")
for ws_id, ws_name in WORKSPACES:
    print(f"  {ws_name} -> {ws_id}")

## Artifact Mapping

Three mappings drive what ends up in the Variable Library:

| Mapping | Produces | Use When |
|---|---|---|
| **`ARTIFACT_MAPPING`** | Scalar GUID variables - `SalesLakehouseId = "abc-123"` | You want a plain item GUID. Use a `dict` value when the same display name resolves to multiple item types (e.g. a Lakehouse + its auto-generated SQLEndpoint both share the Lakehouse's display name but have different GUIDs and types). |
| **`REFERENCE_MAPPING`** | Compound Item Reference variables - `SalesLakehouseRef = {"itemId": "abc-123", "workspaceId": "def-456"}` | You want to bind a pipeline activity or notebook input to a Fabric item directly. Fabric's "Item Reference" variable type expects the `{itemId, workspaceId}` shape, which is tedious to maintain by hand across environments. |
| **`ARTIFACT_DETAIL_MAPPING`** | Per-item metadata variables - `SalesLakehouseSqlConnectionString = "x.datawarehouse.fabric.microsoft.com"` | You need values that only the type-specific GET endpoint returns (SQL connection strings, database names, server FQDNs). Keyed by `(displayName, itemType)` because the same display name can appear under multiple types. Triggers one extra REST call per matched item. |

`REFERENCE_MAPPING` does NOT read the Fabric API separately - it **reuses** a GUID already extracted via `ARTIFACT_MAPPING`. So to produce `SalesLakehouseRef`, you must also have `SalesLakehouseId` in `ARTIFACT_MAPPING`.

`ARTIFACT_DETAIL_MAPPING` **does** call the Fabric API again - one GET per matched item against the type-specific endpoint (e.g. `/lakehouses/{id}`) - because the workspace `/items` listing only returns identity fields, not connection strings or other deep properties. Extend `ITEM_TYPE_URL_SEGMENT` below if you need to support an item type that isn't already listed.

Typical pattern: populate `ARTIFACT_MAPPING` for every item you need, then add `REFERENCE_MAPPING` entries only for items consumed by pipelines/notebooks that require the compound form, and `ARTIFACT_DETAIL_MAPPING` only for items whose connection strings or other deep fields you need to surface as variables.

In [ ]:
# Mapping of Fabric artifact display names to variables.json variable names.
# Value can be a string (matches any item type) or a dict {type: var_name} for
# artifacts that appear as multiple types (e.g. Lakehouse + SQLEndpoint).
#
# Worked examples:
#   "Sales_Lakehouse" is a Lakehouse. Fabric surfaces every Lakehouse as TWO items:
#   the Lakehouse itself and its auto-generated SQLEndpoint. We want GUIDs for both,
#   stored under different variable names, so the value is a dict keyed by type.
#
#   "Sales_Warehouse", "pl_orchestrator", and "nb_ingest" only exist as a single
#   item type each, so a plain string mapping is enough.
ARTIFACT_MAPPING = {
    # "Sales_Lakehouse": {
    #     "Lakehouse":   "SalesLakehouseId",
    #     "SQLEndpoint": "SalesSqlEndpointId",
    # },
    # "Sales_Warehouse": "SalesWarehouseId",
    # "pl_orchestrator": "OrchestratorPipelineId",
    # "nb_ingest":       "IngestNotebookId",
}

# Mapping of ItemReference variable names to their corresponding GUID variable names.
# Used to auto-populate {"itemId": <guid>, "workspaceId": <ws_id>} in value sets.
#
# Worked example:
#   "SalesLakehouseRef" -> "SalesLakehouseId" tells the extractor: create a variable
#   called SalesLakehouseRef whose value is the compound object
#       {"itemId": "<SalesLakehouseId guid>", "workspaceId": "<current workspace guid>"}
#   This is the shape Fabric's "Item Reference" variable type expects - pipelines and
#   notebooks can then bind directly to SalesLakehouseRef instead of juggling two
#   separate variables for itemId and workspaceId.
REFERENCE_MAPPING = {
    # "SalesLakehouseRef":      "SalesLakehouseId",
    # "SalesWarehouseRef":      "SalesWarehouseId",
    # "OrchestratorPipelineRef":"OrchestratorPipelineId",
    # "IngestNotebookRef":      "IngestNotebookId",
}

# Optional per-item metadata fields fetched via the type-specific GET endpoint.
# Keyed by (displayName, itemType) -> {variableName: dotted JSON path in the response}.
# Values are stamped into the variable library alongside the GUIDs from ARTIFACT_MAPPING.
#
# Worked example:
#   ("Sales_Lakehouse", "Lakehouse") with
#       {"SalesLakehouseSqlConnectionString": "properties.sqlEndpointProperties.connectionString"}
#   tells the extractor: after resolving the GUID for the Lakehouse named
#   "Sales_Lakehouse", also GET /workspaces/{wsId}/lakehouses/{id}, walk
#   `properties.sqlEndpointProperties.connectionString`, and stamp the result into
#   the variable library as SalesLakehouseSqlConnectionString.
ARTIFACT_DETAIL_MAPPING = {
    # ("<LakehouseDisplayName>", "Lakehouse"): {
    #     "<LakehouseSqlConnectionStringVar>": "properties.sqlEndpointProperties.connectionString",
    # },
    # ("<SqlDatabaseDisplayName>", "SQLDatabase"): {
    #     "<DatabaseNameVar>":              "properties.databaseName",
    #     "<DatabaseSqlEndpointStringVar>": "properties.serverFqdn",
    # },
    # ("<WarehouseDisplayName>", "Warehouse"): {
    #     "<WarehouseSqlConnectionStringVar>": "properties.connectionString",
    # },
}

# URL segment under /workspaces/{wsId}/ for the per-itemType GET endpoint.
# Casing follows the Fabric REST API docs (SQLDatabases is title-cased).
ITEM_TYPE_URL_SEGMENT = {
    "Lakehouse":   "lakehouses",
    "SQLDatabase": "SQLDatabases",
    "Warehouse":   "warehouses",
}


def get_workspace_items(workspace_id: str) -> List[Dict]:
    """Fetch all items from a workspace via the Fabric REST API."""
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"ERROR: Failed to fetch items for workspace {workspace_id}")
        print(f"  Status: {response.status_code}  Response: {response.text}")
        return []

    return response.json().get("value", [])


def fetch_item_details(workspace_id: str, item_id: str, item_type: str) -> Optional[Dict]:
    """Fetch full item definition via the per-itemType GET endpoint. Returns None on miss."""
    url_segment = ITEM_TYPE_URL_SEGMENT.get(item_type)
    if not url_segment:
        return None
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/{url_segment}/{item_id}"
    response = requests.get(url, headers=HEADERS)
    if response.status_code != 200:
        print(f"  WARNING: Failed to fetch {url_segment}/{item_id}: {response.status_code} {response.text}")
        return None
    return response.json()


def get_nested(data: Dict, dotted_path: str):
    """Walk a dotted JSON path on a dict; return None if any segment is missing."""
    cursor = data
    for key in dotted_path.split("."):
        if not isinstance(cursor, dict):
            return None
        cursor = cursor.get(key)
    return cursor


def extract_guids(
    workspace_id: str, workspace_name: str = ""
) -> Tuple[Dict[str, str], Optional[str]]:
    """Extract GUIDs from a single workspace. Returns (guid_mapping, variable_library_id)."""
    items = get_workspace_items(workspace_id)
    guid_mapping = {}
    variable_library_id = None

    label = f"{workspace_name} ({workspace_id})" if workspace_name else workspace_id
    print(f"\n{label} - {len(items)} items")
    print(f"{'Display Name':<45} {'Type':<20} {'ID'}")
    print("-" * 110)

    for item in sorted(items, key=lambda x: x.get("displayName", "")):
        display_name = item.get("displayName", "")
        item_id = item.get("id", "")
        item_type = item.get("type", "")

        print(f"{display_name:<45} {item_type:<20} {item_id}")

        if display_name in ARTIFACT_MAPPING:
            mapping_value = ARTIFACT_MAPPING[display_name]
            if isinstance(mapping_value, dict):
                if item_type in mapping_value:
                    guid_mapping[mapping_value[item_type]] = item_id
            else:
                guid_mapping[mapping_value] = item_id

        # Truthy check skips null/empty values so an unprovisioned SQL endpoint
        # doesn't overwrite a valid one stamped on a previous run.
        detail_key = (display_name, item_type)
        if detail_key in ARTIFACT_DETAIL_MAPPING:
            details = fetch_item_details(workspace_id, item_id, item_type)
            if details:
                for var_name, json_path in ARTIFACT_DETAIL_MAPPING[detail_key].items():
                    value = get_nested(details, json_path)
                    if value:
                        guid_mapping[var_name] = value

        if display_name == VARIABLE_LIBRARY_NAME and item_type == "VariableLibrary":
            variable_library_id = item_id

    return guid_mapping, variable_library_id

## Extract GUIDs

Runs `extract_guids` for every workspace in scope, prints a per-workspace table of every item (display name, type, GUID), and captures the Variable Library's own GUID from the workspace named in `VARIABLE_LIBRARY_WORKSPACE`.

In [ ]:
# ── Extract GUIDs from all workspaces ────────────────────────────────────────────
all_guid_mappings = {}  # {workspace_id: {var_name: guid}}
variable_library_info = None  # (workspace_id, item_id, workspace_name)

for ws_id, ws_name in WORKSPACES:
    mapping, vl_id = extract_guids(ws_id, ws_name)
    all_guid_mappings[ws_id] = mapping

    # Only use the VL from the explicitly configured workspace
    if vl_id and ws_name == VARIABLE_LIBRARY_WORKSPACE:
        variable_library_info = (ws_id, vl_id, ws_name)

    print(f"\n{'='*110}")
    print(f"GUID MAPPING for {ws_name} ({ws_id}):")
    print(f"{'='*110}")
    for var_name, guid in mapping.items():
        print(f"  {var_name}: {guid}")

if variable_library_info:
    vl_ws_id, vl_item_id, vl_ws_name = variable_library_info
    print(
        f"\nVariable Library '{VARIABLE_LIBRARY_NAME}' found in {vl_ws_name} ({vl_item_id})"
    )
else:
    # Track as a structured failure - this blocks the update flow even though
    # GUID extraction itself succeeded. The update cell below will check and
    # skip with a matching entry; surfacing it here gives the summary context.
    reason = (
        f"Variable Library '{VARIABLE_LIBRARY_NAME}' not found in workspace "
        f"'{VARIABLE_LIBRARY_WORKSPACE}'. Check VARIABLE_LIBRARY_WORKSPACE config."
    )
    print(f"\nWARNING: {reason}")
    results["failed"].append({"name": "variable-library-lookup", "error": reason})

## Update Variable Library

When `UPDATE_VALUE_SETS = True`, fetches the Variable Library definition via `getDefinition`, applies the extracted GUIDs to `variables.json` (the `default` value set) and any targeted `valueSets/<name>.json` overrides, then posts the full definition back via `updateDefinition`. Every part of the original definition is re-sent - the Fabric API replaces the entire definition on update, so omitting any part would delete it.

In [ ]:
# ── Update Variable Library via Fabric REST API (optional) ────────────────────────
def apply_guid_updates(variables: List[Dict], ws_id: str, ws_name: str,
                       guid_mapping: Dict[str, str], label: str) -> List[str]:
    """
    Apply extracted GUIDs to a list of variable entries in place.

    Works for both variables.json entries (fields: 'type', 'note') and value
    set override entries. Returns the list of variable names that were updated
    or appended. This is a pure in-place mutation helper - orchestration is in
    the caller below.

    :param variables: The mutable list of variable entries to update.
    :param ws_id: Workspace GUID to pair with WorkspaceId / ItemReference values.
    :param ws_name: Workspace display name to pair with WorkspaceName values.
    :param guid_mapping: Extracted {var_name: guid} map for the target workspace.
    :param label: Descriptive label for log output (e.g. "default (variables.json)").
    :returns: List of variable names that were updated or added.
    """
    updates_made = []
    existing_names = {v["name"] for v in variables}

    print(f"\n  {label}:")
    print(f"    Extracted GUIDs ({len(guid_mapping)}): {list(guid_mapping.keys())}")
    print(f"    Existing variables ({len(existing_names)}): {sorted(existing_names)}")

    for entry in variables:
        var_name = entry.get("name")
        old_value = entry.get("value")
        new_value = None

        if var_name == "WorkspaceId":
            new_value = ws_id
        elif var_name == "WorkspaceName":
            new_value = ws_name
        elif var_name in guid_mapping:
            new_value = guid_mapping[var_name]
        elif var_name in REFERENCE_MAPPING:
            item_guid_var = REFERENCE_MAPPING[var_name]
            if item_guid_var in guid_mapping:
                new_value = {"itemId": guid_mapping[item_guid_var], "workspaceId": ws_id}

        if new_value is not None:
            if old_value != new_value:
                entry["value"] = new_value
                updates_made.append(var_name)
                print(f"    CHANGED {var_name}: {old_value} -> {new_value}")
            else:
                print(f"    MATCH   {var_name}: {old_value}")
        else:
            print(f"    SKIP    {var_name}: not in extracted GUIDs")

    # Add missing workspace identity entries
    for name, value in [("WorkspaceId", ws_id), ("WorkspaceName", ws_name)]:
        if name not in existing_names:
            variables.append({"name": name, "value": value})
            updates_made.append(name)
            print(f"    ADDED   {name}: {value}")

    # Add missing entries for extracted GUIDs not already present
    for var_name, guid in guid_mapping.items():
        if var_name not in existing_names:
            variables.append({"name": var_name, "value": guid})
            updates_made.append(var_name)
            print(f"    ADDED   {var_name}: {guid}")

    for ref_name, guid_var in REFERENCE_MAPPING.items():
        if ref_name not in existing_names and guid_var in guid_mapping:
            ref_value = {"itemId": guid_mapping[guid_var], "workspaceId": ws_id}
            variables.append({"name": ref_name, "value": ref_value})
            updates_made.append(ref_name)
            print(f"    ADDED   {ref_name}: {ref_value}")

    if updates_made:
        print(f"    >> {len(updates_made)} variable(s) updated/added")
    else:
        print("    >> already up to date")

    return updates_made


if not UPDATE_VALUE_SETS:
    # User explicitly opted out - not a failure, just nothing to do.
    print("\nSkipping value set updates (set UPDATE_VALUE_SETS = True to enable).")
    for vs_name in VALUE_SETS_TO_UPDATE:
        results["skipped"].append({"name": vs_name, "reason": "UPDATE_VALUE_SETS=False"})

elif not variable_library_info:
    # The extract cell already recorded the VL-lookup failure. Record each
    # targeted value-set as skipped so the summary shows what was attempted.
    print(
        f"ERROR: Cannot update - Variable Library '{VARIABLE_LIBRARY_NAME}' not found."
    )
    for vs_name in VALUE_SETS_TO_UPDATE:
        results["skipped"].append({"name": vs_name, "reason": "variable library not found"})

elif not VALUE_SETS_TO_UPDATE:
    # Empty target map - no work defined. Also not a failure.
    print("No targets in VALUE_SETS_TO_UPDATE - nothing to do.")

else:
    vl_ws_id, vl_item_id, vl_ws_name = variable_library_info
    print(f"Fetching definition for '{VARIABLE_LIBRARY_NAME}' from {vl_ws_name}...")

    # GET current Variable Library definition
    get_url = (
        f"{FABRIC_API_BASE}/workspaces/{vl_ws_id}"
        f"/VariableLibraries/{vl_item_id}/getDefinition"
    )
    response = poll_lro(requests.post(get_url, headers=HEADERS), HEADERS)

    if response.status_code != 200:
        # Tier-1 condition - can't proceed without the definition. Raise
        # regardless of STRICT; there is no per-value-set outcome to collect.
        raise RuntimeError(
            f"Failed to get definition for '{VARIABLE_LIBRARY_NAME}': "
            f"{response.status_code} {response.text}"
        )

    definition = response.json()["definition"]

    # Build workspace name -> (ws_id, guid_mapping) lookup
    ws_lookup = {
        ws_name: (ws_id, all_guid_mappings.get(ws_id, {}))
        for ws_id, ws_name in WORKSPACES
    }

    # Validate all targets in VALUE_SETS_TO_UPDATE reference known workspaces.
    # Any target pointing at an unknown workspace is skipped - we cannot
    # produce GUID values for it without having scanned it.
    for target, ws_name in VALUE_SETS_TO_UPDATE.items():
        if ws_name not in ws_lookup:
            reason = f"workspace '{ws_name}' not in WORKSPACE_NAMES"
            print(
                f"WARNING: VALUE_SETS_TO_UPDATE['{target}'] references "
                f"{reason}. Skipping."
            )
            results["skipped"].append({"name": target, "reason": reason})

    updated_parts = []
    touched_targets = set()  # value-set names that matched a part in the definition

    for part in definition["parts"]:
        path = part["path"]
        raw = base64.b64decode(part["payload"]).decode("utf-8")

        target_key = None
        if path == "variables.json" and "default" in VALUE_SETS_TO_UPDATE:
            target_key = "default"
        elif path.startswith("valueSets/"):
            # Extract value set name from path (e.g. "valueSets/Env-1D.json" -> "Env-1D")
            vs_name = path.split("/")[-1].replace(".json", "")
            if vs_name in VALUE_SETS_TO_UPDATE:
                target_key = vs_name

        if target_key is not None:
            touched_targets.add(target_key)
            target_ws_name = VALUE_SETS_TO_UPDATE[target_key]
            if target_ws_name in ws_lookup:
                ws_id, guid_mapping = ws_lookup[target_ws_name]
                try:
                    data = json.loads(raw)

                    if path == "variables.json":
                        updates = apply_guid_updates(
                            data["variables"], ws_id, target_ws_name,
                            guid_mapping, f"default (variables.json) <- {target_ws_name}"
                        )
                    else:
                        # Ensure variableOverrides key exists so appends are captured
                        if "variableOverrides" not in data:
                            data["variableOverrides"] = []
                        updates = apply_guid_updates(
                            data["variableOverrides"],
                            ws_id, target_ws_name, guid_mapping,
                            f"{path} <- {target_ws_name}"
                        )

                    if updates:
                        results["succeeded"].append(target_key)
                    else:
                        results["skipped"].append({
                            "name": target_key,
                            "reason": "already up to date",
                        })

                    raw = json.dumps(data, indent=2)

                except Exception as ex:
                    results["failed"].append({
                        "name": target_key,
                        "error": f"apply_guid_updates: {ex}",
                    })
            # else: already recorded as skipped during validation above

        encoded = base64.b64encode(raw.encode("utf-8")).decode("utf-8")
        updated_parts.append(
            {"path": path, "payload": encoded, "payloadType": "InlineBase64"}
        )

    # Targets configured but not present in the definition - flag as skipped
    # so the user sees that their config didn't match any part.
    for target in VALUE_SETS_TO_UPDATE:
        if target not in touched_targets and target not in {
            e["name"] for e in results["skipped"]
        } and target not in {e["name"] for e in results["failed"]}:
            results["skipped"].append({
                "name": target,
                "reason": f"no matching part in definition (expected path 'variables.json' or 'valueSets/{target}.json')",
            })

    # POST updated definition back
    if results["succeeded"]:
        try:
            update_url = (
                f"{FABRIC_API_BASE}/workspaces/{vl_ws_id}"
                f"/VariableLibraries/{vl_item_id}/updateDefinition"
            )
            body = {"definition": {"parts": updated_parts}}
            response = poll_lro(
                requests.post(update_url, headers=HEADERS, json=body), HEADERS
            )

            if response.status_code in (200, 202):
                print(
                    f"\nVariable Library '{VARIABLE_LIBRARY_NAME}' updated successfully."
                )
            else:
                # Move every previously-succeeded value set into failed - the
                # API call that would persist their changes didn't land.
                err = f"updateDefinition: {response.status_code} {response.text}"
                for vs_name in list(results["succeeded"]):
                    results["succeeded"].remove(vs_name)
                    results["failed"].append({"name": vs_name, "error": err})
                print(f"\nERROR: Update failed: {response.status_code} {response.text}")
        except Exception as ex:
            err = f"updateDefinition: {ex}"
            for vs_name in list(results["succeeded"]):
                results["succeeded"].remove(vs_name)
                results["failed"].append({"name": vs_name, "error": err})
            print(f"\nERROR: Update failed: {ex}")
    else:
        print("\nAll targets are already up to date or skipped. No update needed.")

# ── Summary ──────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("  Summary")
print("─" * 80)
print(f"  Succeeded: {len(results['succeeded'])}")
print(f"  Skipped:   {len(results['skipped'])}")
print(f"  Failed:    {len(results['failed'])}")

for name in results["succeeded"]:
    print(f"    OK   {name}")
for entry in results["skipped"]:
    print(f"    SKIP {entry['name']}: {entry['reason']}")
for entry in results["failed"]:
    print(f"    FAIL {entry['name']}: {entry['error']}")

print("─" * 80)

if STRICT and results["failed"]:
    raise RuntimeError(
        f"STRICT mode: {len(results['failed'])} target(s) failed. "
        f"See summary above."
    )